In [6]:
# Клонируем репозиторий CatVTON
!git clone https://github.com/Zheng-Chong/CatVTON.git
%cd CatVTON

fatal: destination path 'CatVTON' already exists and is not an empty directory.
/content/CatVTON


In [9]:
# Устанавливаем зависимости из requirements.txt
!pip install -r requirements.txt

# Обновляем accelerate до последней версии (>=0.32.0) и ставим стабильный diffusers
!pip install -U accelerate diffusers==0.30.3 peft huggingface_hub

  Cloning https://github.com/huggingface/diffusers.git to /tmp/pip-req-build-0uv7n_8e
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/diffusers.git /tmp/pip-req-build-0uv7n_8e
  Resolved https://github.com/huggingface/diffusers.git to commit c8eba433adf1f90d7fcc70092562ea50789ee8fb
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached accelerate-0.31.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-0.31.0-py3-none-any.whl (309 kB)
  Created wheel for diffusers: filename=diffusers-0.38.0.dev0-py3-none-any.whl size=5245992 sha256=19d3b31b271cab73edb468e908f111ec7eaf371f81ca98ae53258752791ff1ca
  Stored in directory: /tmp/pip-ephem-wheel-cache-kmb5or6y/wheels/23/0f/7d/f97813d265ed0e599a78d83afd4e1925740896ca79b46cccfd
Successfully built diffusers
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.30.3
    Uninstalling 

  Using cached diffusers-0.30.3-py3-none-any.whl.metadata (18 kB)
Using cached diffusers-0.30.3-py3-none-any.whl (2.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 34.8 MB/s eta 0:00:00
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.38.0.dev0
    Uninstalling diffusers-0.38.0.dev0:
      Successfully uninstalled diffusers-0.38.0.dev0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 0.31.0
    Uninstalling accelerate-0.31.0:
      Successfully uninstalled accelerate-0.31.0


In [1]:
# Возвращаем стабильный PyTorch 2.4.0 и совместимый xformers (0.0.27.post2)
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121
!pip install xformers==0.0.27.post2

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 794.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 14.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.5/209.5 MB 5.5 MB/s eta 0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 3.6.0
    Uninstalling triton-3.6.0:
      Successfully uninstalled triton-3.6.0
  Attempting uninstall: torch
    Found existing installation: torch 2.11.0
    Uninstalling torch-2.11.0:
      Successfully uninstalled torch-2.11.0
  Attempting uninstall: torchaudio
    Found existing installation: torchaudio 2.10.0+cu128
    Uninstalling torchaudio-2.10.0+cu128:
      Successfully uninstalled torchaudio-2.10.0+cu128
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xformers 0.0.35 requires torch>=2

In [5]:
import urllib.request

# test human-model image (front view)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/Zheng-Chong/CatVTON/main/resource/demo/example/person/men/model_5.png",
    "person.png"
)

# test garment image
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/Zheng-Chong/CatVTON/main/resource/demo/example/condition/upper/24083449_54173465_2048.jpg",
    "garment.jpg"
)

('garment.jpg', <http.client.HTTPMessage at 0x7c06082fc110>)

In [3]:
%cd CatVTON

/content/CatVTON


In [4]:
%ls

app_flux.py  detectron2/   model/                       resource/
app_p2p.py   eval.py       preprocess_agnostic_mask.py  utils.py
app.py       index.html    __pycache__/
densepose/   inference.py  README.md
densepose_/  LICENSE       requirements.txt


In [5]:
import torch
from PIL import Image
from huggingface_hub import snapshot_download

# imports from CatVTON
from model.cloth_masker import AutoMasker
from model.pipeline import CatVTONPipeline

/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")


In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [7]:
weight_dtype = torch.bfloat16

base_model_path = "runwayml/stable-diffusion-inpainting"
catvton_model_path = "zhengchong/CatVTON"

pipeline = CatVTONPipeline(
    base_ckpt=base_model_path,
    attn_ckpt=catvton_model_path,
    attn_ckpt_version="mix",
    weight_dtype=weight_dtype,
    device=device,
    skip_safety_check=True
)

try:
    pipeline.enable_xformers_memory_efficient_attention()
    print("Оптимизация xformers успешно включена!")
except Exception:
    print("xformers not installed, skip memory optimization")

# Дополнительные оптимизации для экономии VRAM (полезно для 8GB GPU)
try:
    pipeline.vae.enable_slicing()
    pipeline.vae.enable_tiling()
    print("Оптимизации VAE (slicing/tiling) включены!")
except Exception as e:
    print(f"Не удалось включить оптимизации VAE: {e}")

An error occurred while trying to fetch runwayml/stable-diffusion-inpainting: runwayml/stable-diffusion-inpainting does not appear to have a file named diffusion_pytorch_model.safetensors.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Downloaded zhengchong/CatVTON to /root/.cache/huggingface/hub/models--zhengchong--CatVTON/snapshots/2969fcf85fe62f2036605716f0b56f0b81d01d79
xformers not installed, skip memory optimization


In [8]:
import os
from huggingface_hub import snapshot_download

# Скачиваем основной репозиторий с весами (или берем из кэша)
repo_path = snapshot_download(repo_id="zhengchong/CatVTON")

# Инициализация модуля для автоматического создания масок с использованием путей внутри основного репозитория
automasker = AutoMasker(
    densepose_ckpt=os.path.join(repo_path, "DensePose"),
    schp_ckpt=os.path.join(repo_path, "SCHP"),
    device=device
)

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/content/CatVTON/model/SCHP/__init__.py:93: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(ckpt_path, map_location='cpu')['state_dict']


In [9]:
import matplotlib.pyplot as plt
from PIL import Image
import torch
import gc

# Очищаем память перед запуском
torch.cuda.empty_cache()
gc.collect()

# 1. Загрузка изображений (пути к файлам, которые мы скачали ранее)
person_path = "/content/person.png"
garment_path = "/content/garment.jpg"

person_image = Image.open(person_path).convert("RGB")
garment_image = Image.open(garment_path).convert("RGB")

# Уменьшаем разрешение до 512x768, чтобы избежать OOM на бесплатном T4 GPU
target_size = (512, 768)
person_image = person_image.resize(target_size)
garment_image = garment_image.resize(target_size)

# 2. Генерация маски
print("Генерация маски...")
mask_result = automasker(person_image)
mask = mask_result['mask']

# 3. Запуск инференса
print("Запуск генерации (примерки). Это может занять пару минут...")
result_image = pipeline(
    image=person_image,
    condition_image=garment_image,
    mask=mask,
    num_inference_steps=50,
    guidance_scale=2.5
)[0]

# 4. Отображение результатов
fig, axs = plt.subplots(1, 4, figsize=(20, 7))

axs[0].imshow(person_image)
axs[0].set_title("Исходное фото")
axs[0].axis('off')

axs[1].imshow(garment_image)
axs[1].set_title("Одежда")
axs[1].axis('off')

axs[2].imshow(mask, cmap='gray')
axs[2].set_title("Сгенерированная маска")
axs[2].axis('off')

axs[3].imshow(result_image)
axs[3].set_title("Результат примерки")
axs[3].axis('off')

plt.tight_layout()
plt.show()

Генерация маски...


/usr/local/lib/python3.12/dist-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3609.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Запуск генерации (примерки). Это может занять пару минут...


  0%|          | 0/50 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 18.00 GiB. GPU 0 has a total capacity of 14.56 GiB of which 11.50 GiB is free. Including non-PyTorch memory, this process has 3.06 GiB memory in use. Of the allocated memory 2.86 GiB is allocated by PyTorch, and 78.36 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)